### Flights Scraper Sky API

In [1]:
import sys
import os

# Check Python version
print(f"Python Version: `{sys.version}`")  # Detailed version info
print(f"Base Python location: `{sys.base_prefix}`")
print(f"Current Environment location: `{os.path.basename(sys.prefix)}`", end='\n\n')

Python Version: `3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]`
Base Python location: `C:\Users\LMT\AppData\Local\Programs\Python\Python312`
Current Environment location: `.venv`



In [2]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import json
import time

In [3]:
# Load environment variables
load_dotenv()

True

In [14]:
class FlightsScraperSky:
    def __init__(self, rapidapi_key):
        self.api_key = rapidapi_key
        self.base_url = "https://flights-sky.p.rapidapi.com"
        self.headers = {
            "X-RapidAPI-Key": self.api_key,
            "X-RapidAPI-Host": "flights-sky.p.rapidapi.com"
        }
    
    def search_airport(self, query):
        """Search for airports"""
        url = f"{self.base_url}/flights/auto-complete"
        
        params = {
            "query": query,
            "market": "UK",
            "locale": "en-GB"
        }
        
        print(f"{url} | Searching airports with query: {query}")
        
        try:
            response = requests.get(url, headers=self.headers, params=params)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"Error searching airports: {e}")
            return None
    
    def get_location_ids(self, city_name):
        """Get entityId for flight search"""
        results = self.search_airport(city_name)
        
        if not results or not results.get('status'):
            return None
        
        data = results.get('data', [])
        if not data:
            return None
        
        first = data[0]
        presentation = first.get('presentation', {})
        
        return {
            'entityId': presentation.get('id'),
            'skyId': presentation.get('skyId'),
            'name': presentation.get('title')
        }
    
    def search_roundtrip(self, from_entity_id, to_entity_id, depart_date, return_date):
        """Search for round-trip flights"""
        url = f"{self.base_url}/flights/search-roundtrip"
        
        params = {
            "fromEntityId": from_entity_id,
            "toEntityId": to_entity_id,
            "departDate": depart_date,
            "returnDate": return_date,
            "market": "UK",
            "locale": "en-GB",
            "currency": "GBP",
            "adults": "1",
            "cabinClass": "economy"
        }
        
        print(f"{url} | Searching roundtrip flights from {from_entity_id} to {to_entity_id}")
        
        try:
            print(f"\nSearching: {from_entity_id} -> {to_entity_id}")
            print(f"Dates: {depart_date} to {return_date}")
            
            response = requests.get(url, headers=self.headers, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"Error searching flights: {e}")
            return None
    
    def check_if_incomplete(self, results):
        """Check if results need polling"""
        if not results or not results.get('status'):
            return True
        
        context = results.get('data', {}).get('context', {})
        status = context.get('status', 'unknown')
        
        print(f"Status: {status}")
        return status == 'incomplete'
    
    def poll_incomplete_results(self, session_id):
        """Poll for complete results"""
        url = f"{self.base_url}/flights/search-incomplete"
        
        params = {
            "sessionId": session_id,
            "market": "UK",
            "locale": "en-GB",
            "currency": "GBP"
        }
        
        print(f"{url} | Polling for session ID: {session_id}")
        
        max_attempts = 10
        
        for attempt in range(max_attempts):
            print(f"Polling {attempt + 1}/{max_attempts}...")
            
            try:
                time.sleep(3)  # Wait before polling
                
                response = requests.get(url, headers=self.headers, params=params, timeout=30)
                response.raise_for_status()
                results = response.json()
                
                if not self.check_if_incomplete(results):
                    print("Complete!")
                    return results
                
            except Exception as e:
                print(f"Polling error: {e}")
                return None
        
        print("Max attempts reached")
        return None

    def extract_transit_details(self, segments):
        """Extract transit country codes and layover durations"""
        if not segments or len(segments) <= 1:
            return {
                'countries': [],
                'airports': [],
                'layover_hours': []
            }
        
        countries = []
        airports = []
        layover_hours = []
        
        # Each connection point (except final destination)
        for i in range(len(segments) - 1):
            # Transit airport
            transit_airport = segments[i]['destination']['flightPlaceId']
            airports.append(transit_airport)
            
            # Transit country - get from parent country
            # Need to look this up from dictionaries
            # For now, we'll extract from the destination field if available
            dest_info = segments[i].get('destination', {})
            country = dest_info.get('country', 'UNKNOWN')
            countries.append(country)
            
            # Calculate layover
            arrival = segments[i]['arrival']
            next_departure = segments[i + 1]['departure']
            
            arr_dt = pd.to_datetime(arrival)
            dep_dt = pd.to_datetime(next_departure)
            layover = (dep_dt - arr_dt).total_seconds() / 3600
            layover_hours.append(round(layover, 1))
        
        return {
            'countries': countries,
            'airports': airports,
            'layover_hours': layover_hours
        }
      
    def parse_flights(self, results):
        """Parse flight results into pandas DataFrame"""
        if not results or not results.get('status'):
            return pd.DataFrame()
        
        itineraries = results.get('data', {}).get('itineraries', [])
        
        if not itineraries:
            return pd.DataFrame()
        
        flights_data = []
        
        for itin in itineraries:
            try:
                # Basic info
                flight_id = itin.get('id')
                price = itin.get('price', {}).get('raw', 0)
                
                legs = itin.get('legs', [])
                
                # Outbound leg (first)
                outbound = legs[0] if len(legs) > 0 else {}
                outbound_segs = outbound.get('segments', [])
                
                # Return leg (second)
                return_leg = legs[1] if len(legs) > 1 else {}
                return_segs = return_leg.get('segments', [])
                
                # Extract transit countries and layovers
                outbound_transits = self.extract_transit_details(outbound_segs)
                return_transits = self.extract_transit_details(return_segs)
                
                # Compile all data
                flight_row = {
                    'flight_id': flight_id,
                    'price_gbp': price,
                    
                    # Outbound info
                    'out_origin': outbound.get('origin', {}).get('displayCode'),
                    'out_destination': outbound.get('destination', {}).get('displayCode'),
                    'out_departure': outbound.get('departure'),
                    'out_arrival': outbound.get('arrival'),
                    'out_duration_min': outbound.get('durationInMinutes'),
                    'out_stops': outbound.get('stopCount'),
                    'out_airline': ', '.join([c.get('name', '') for c in outbound.get('carriers', {}).get('marketing', [])]),
                    'out_transit_countries': outbound_transits['countries'],
                    'out_transit_airports': outbound_transits['airports'],
                    'out_layover_hours': outbound_transits['layover_hours'],
                    
                    # Return info
                    'ret_origin': return_leg.get('origin', {}).get('displayCode'),
                    'ret_destination': return_leg.get('destination', {}).get('displayCode'),
                    'ret_departure': return_leg.get('departure'),
                    'ret_arrival': return_leg.get('arrival'),
                    'ret_duration_min': return_leg.get('durationInMinutes'),
                    'ret_stops': return_leg.get('stopCount'),
                    'ret_airline': ', '.join([c.get('name', '') for c in return_leg.get('carriers', {}).get('marketing', [])]),
                    'ret_transit_countries': return_transits['countries'],
                    'ret_transit_airports': return_transits['airports'],
                    'ret_layover_hours': return_transits['layover_hours'],
                    
                    # Additional
                    'total_duration_min': (outbound.get('durationInMinutes', 0) + 
                                        return_leg.get('durationInMinutes', 0)),
                    'total_stops': outbound.get('stopCount', 0) + return_leg.get('stopCount', 0),
                }
                
                flights_data.append(flight_row)
                
            except Exception as e:
                print(f"Error parsing flight: {e}")
                continue
        
        df = pd.DataFrame(flights_data)
        
        # Add calculated columns
        if not df.empty:
            df['out_departure_date'] = pd.to_datetime(df['out_departure']).dt.date
            df['ret_departure_date'] = pd.to_datetime(df['ret_departure']).dt.date
            df['trip_length_days'] = (pd.to_datetime(df['ret_departure']) - 
                                    pd.to_datetime(df['out_departure'])).dt.days
        
        return df

In [15]:
def test_api():
    """Test the API with correct structure"""
    api_key = os.getenv('FLIGHTS_API_KEY')
    
    if not api_key:
        print("Error: FLIGHTS_API_KEY not in .env")
        return
    
    print("="*60)
    print("TESTING FLIGHTS SCRAPER SKY API")
    print("="*60)
    
    scanner = FlightsScraperSky(api_key)
    
    print("\nStep 1: Get London location ID...")
    london = scanner.get_location_ids("London")
    
    if not london:
        print("Failed to get London")
        return
    
    print(f"Success: {london['name']} (entityId: {london['entityId']})")
    
    print("\nStep 2: Get Ho Chi Minh location ID...")
    hcmc = scanner.get_location_ids("Ho Chi Minh")
    
    if not hcmc:
        print("Failed to get Ho Chi Minh")
        return
    
    print(f"Success: {hcmc['name']} (entityId: {hcmc['entityId']})")
    
    print("\nStep 3: Search flights...")
    results = scanner.search_roundtrip(
        from_entity_id=london['entityId'],
        to_entity_id=hcmc['entityId'],
        depart_date="2025-12-20",
        return_date="2026-01-10"
    )
    
    if not results:
        print("Failed to get results")
        return
    
    if scanner.check_if_incomplete(results):
        print("\nResults incomplete, polling...")
        session_id = results.get('data', {}).get('context', {}).get('sessionId')
        if session_id:
            results = scanner.poll_incomplete_results(session_id)
        else:
            print("No session ID found!")
            return
    
    if not results:
        print("Failed to get complete results")
        return
    
    return scanner, results


def display_flights(scanner, results):
    print("\nParse results...")

    # Get results
    flights_df = scanner.parse_flights(results)

    # Display summary
    print(f"\nTotal flights: {len(flights_df)}")
    print(f"\nPrice range: GBP {flights_df['price_gbp'].min():.2f} - {flights_df['price_gbp'].max():.2f}")
    print(f"Average price: GBP {flights_df['price_gbp'].mean():.2f}")

    # Show top 5 cheapest
    print("\n=== TOP 5 CHEAPEST ===")
    display(flights_df.nsmallest(5, 'price_gbp')[
        ['price_gbp', 'out_airline', 'out_transit_airports', 'ret_transit_airports', 'total_stops']
    ])

    # Filter by transit country
    print("\n=== FLIGHTS VIA DELHI (DEL) ===")
    delhi_flights = flights_df[flights_df['out_transit_airports'].apply(lambda x: 'DEL' in x)]
    display(delhi_flights[['price_gbp', 'out_airline', 'out_transit_airports']])

    # Export to CSV for analysis
    flights_df.to_csv('flight_options.csv', index=False)
    print("\nData saved to flight_options.csv")
        
    return flights_df

In [16]:
scanner, results = test_api()

TESTING FLIGHTS SCRAPER SKY API

Step 1: Get London location ID...
https://flights-sky.p.rapidapi.com/flights/auto-complete | Searching airports with query: London
Success: London (entityId: eyJlIjoiMjc1NDQwMDgiLCJzIjoiTE9ORCIsImgiOiIyNzU0NDAwOCIsInQiOiJDSVRZIn0=)

Step 2: Get Ho Chi Minh location ID...
https://flights-sky.p.rapidapi.com/flights/auto-complete | Searching airports with query: Ho Chi Minh
Success: Ho Chi Minh City (entityId: eyJlIjoiOTU2NzMzNzkiLCJzIjoiU0dOIiwiaCI6IjI3NTQ2MzI5IiwidCI6IkFJUlBPUlQifQ==)

Step 3: Search flights...
https://flights-sky.p.rapidapi.com/flights/search-roundtrip | Searching roundtrip flights from eyJlIjoiMjc1NDQwMDgiLCJzIjoiTE9ORCIsImgiOiIyNzU0NDAwOCIsInQiOiJDSVRZIn0= to eyJlIjoiOTU2NzMzNzkiLCJzIjoiU0dOIiwiaCI6IjI3NTQ2MzI5IiwidCI6IkFJUlBPUlQifQ==

Searching: eyJlIjoiMjc1NDQwMDgiLCJzIjoiTE9ORCIsImgiOiIyNzU0NDAwOCIsInQiOiJDSVRZIn0= -> eyJlIjoiOTU2NzMzNzkiLCJzIjoiU0dOIiwiaCI6IjI3NTQ2MzI5IiwidCI6IkFJUlBPUlQifQ==
Dates: 2025-12-20 to 2026-01-10
Status

In [17]:
flights_df = display_flights(scanner, results)


Parse results...

Total flights: 859

Price range: GBP 934.00 - 3614.00
Average price: GBP 2242.58

=== TOP 5 CHEAPEST ===


,price_gbp,out_airline,out_transit_airports,ret_transit_airports,total_stops
54,934.00,"Ryanair, China Eastern","[BCN, PVG, KMG]","[HYD, DEL, CDG]",6
18,1353.62,Air India,[DEL],[DEL],2
64,1353.62,Air India,[DEL],[DEL],2
141,1353.62,Air India,[DEL],[DEL],2
0,1428.62,Air India,[DEL],[DEL],2



=== FLIGHTS VIA DELHI (DEL) ===


,price_gbp,out_airline,out_transit_airports
0,1428.62,Air India,[DEL]
9,1773.83,Air India,[DEL]
13,1773.52,Air India,[DEL]
18,1353.62,Air India,[DEL]
21,1825.00,Air India,[DEL]
...,...,...,...
837,2305.82,Air India,[DEL]
843,2094.90,Air India,[DEL]
844,2004.72,Air India,[DEL]
853,2212.90,Air India,[DEL]



Data saved to flight_options.csv


In [18]:
flights_df

,flight_id,price_gbp,out_origin,out_destination,out_departure,out_arrival,out_duration_min,out_stops,out_airline,out_transit_countries,...,ret_stops,ret_airline,ret_transit_countries,ret_transit_airports,ret_layover_hours,total_duration_min,total_stops,out_departure_date,ret_departure_date,trip_length_days
0,13554-2512202105--32672-1-16240-2512211955|162...,1428.62,LHR,SGN,2025-12-20T21:05:00,2025-12-21T19:55:00,950,1,Air India,[India],...,1,Air India,[India],[DEL],[5.3],2245,2,2025-12-20,2026-01-10,20
1,13542-2512202025--32348-1-16240-2512211935|162...,1980.67,LGW,SGN,2025-12-20T20:25:00,2025-12-21T19:35:00,970,1,Emirates,[United Arab Emirates],...,1,Emirates,[United Arab Emirates],[DXB],[3.4],2095,2,2025-12-20,2026-01-10,21
2,13542-2512202025--32348-1-16240-2512211935|162...,1980.97,LGW,SGN,2025-12-20T20:25:00,2025-12-21T19:35:00,970,1,Emirates,[United Arab Emirates],...,1,Emirates,[United Arab Emirates],[DXB],[3.3],2095,2,2025-12-20,2026-01-10,21
3,13554-2512202120--32348-1-16240-2512211935|162...,2008.08,LHR,SGN,2025-12-20T21:20:00,2025-12-21T19:35:00,915,1,Emirates,[United Arab Emirates],...,1,Emirates,[United Arab Emirates],[DXB],[3.4],2040,2,2025-12-20,2026-01-10,21
4,13554-2512202120--32348-1-16240-2512211935|162...,2008.38,LHR,SGN,2025-12-20T21:20:00,2025-12-21T19:35:00,915,1,Emirates,[United Arab Emirates],...,1,Emirates,[United Arab Emirates],[DXB],[3.3],2040,2,2025-12-20,2026-01-10,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
854,13554-2512202205--32456-1-16240-2512220020|162...,2744.72,LHR,SGN,2025-12-20T22:05:00,2025-12-22T00:20:00,1155,1,Cathay Pacific,[Hong Kong],...,1,Malaysia Airlines,[Malaysia],[KUL],[1.5],2210,2,2025-12-20,2026-01-10,20
855,13554-2512201300--32672-1-16240-2512211955|162...,2306.32,LHR,SGN,2025-12-20T13:00:00,2025-12-21T19:55:00,1435,1,Air India,[India],...,2,Air France,"[Vietnam, France]","[HAN, CDG]","[2.2, 1.0]",2630,3,2025-12-20,2026-01-10,21
856,13554-2512201550--32348-1-16240-2512211935|162...,2194.41,LHR,SGN,2025-12-20T15:50:00,2025-12-21T19:35:00,1245,1,Emirates,[United Arab Emirates],...,2,"Vietnam Airlines, Oman Air","[Thailand, Oman]","[BKK, MCT]","[6.9, 2.4]",2765,3,2025-12-20,2026-01-10,20
857,13554-2512201455--31939-1-16240-2512211305|162...,3058.17,LHR,SGN,2025-12-20T14:55:00,2025-12-21T13:05:00,910,1,Qatar Airways,[Qatar],...,1,Qatar Airways,[Qatar],[DOH],[2.3],1985,2,2025-12-20,2026-01-10,20
